In [ ]:
!pip install -q transformers datasets rouge_score

In [ ]:
import torch
from datasets import load_dataset
from transformers import BartTokenizer, BartForConditionalGeneration
from torch.utils.data import DataLoader
from torch.optim import AdamW
from rouge_score import rouge_scorer

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)
if device == "cuda":
    print("GPU name:", torch.cuda.get_device_name(0))

Using device: cuda
GPU name: Tesla T4


In [ ]:
from datasets import load_dataset


ds = load_dataset("abisee/cnn_dailymail", "1.0.0")


train_dataset = ds["train"].select(range(10000))
test_dataset = ds["test"].select(range(1000))

print(train_dataset[0])


{'article': 'LONDON, England (Reuters) -- Harry Potter star Daniel Radcliffe gains access to a reported £20 million ($41.1 million) fortune as he turns 18 on Monday, but he insists the money won\'t cast a spell on him. Daniel Radcliffe as Harry Potter in "Harry Potter and the Order of the Phoenix" To the disappointment of gossip columnists around the world, the young actor says he has no plans to fritter his cash away on fast cars, drink and celebrity parties. "I don\'t plan to be one of those people who, as soon as they turn 18, suddenly buy themselves a massive sports car collection or something similar," he told an Australian interviewer earlier this month. "I don\'t think I\'ll be particularly extravagant. "The things I like buying are things that cost about 10 pounds -- books and CDs and DVDs." At 18, Radcliffe will be able to gamble in a casino, buy a drink in a pub or see the horror film "Hostel: Part II," currently six places below his number one movie on the UK box office char

In [ ]:
tokenizer = BartTokenizer.from_pretrained("facebook/bart-base")


Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

In [ ]:
def encode_batch(batch):
    inputs = tokenizer(batch["article"], truncation=True, padding="max_length", max_length=512)
    targets = tokenizer(batch["highlights"], truncation=True, padding="max_length", max_length=128)
    return {"input_ids": inputs["input_ids"], "labels": targets["input_ids"]}


encoded_train = train_dataset.map(encode_batch, batched=True, num_proc=2)
encoded_test = test_dataset.map(encode_batch, batched=True, num_proc=2)


Map (num_proc=2):   0%|          | 0/10000 [00:00<?, ? examples/s]

Map (num_proc=2):   0%|          | 0/1000 [00:00<?, ? examples/s]

In [ ]:
encoded_train.set_format(type="torch", columns=["input_ids", "labels"])

In [ ]:
train_loader = DataLoader(encoded_train, batch_size=4, shuffle=True)

In [ ]:
optimizer = AdamW(model.parameters(), lr=5e-5)

In [ ]:
model.train()
for epoch in range(1):
    total_loss = 0
    for step, batch in enumerate(train_loader, 1):
        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(input_ids=input_ids, labels=labels)
        loss = outputs.loss
        total_loss += loss.item()

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if step % 500 == 0:
            print(f"Epoch {epoch+1}, Step {step}, Avg Loss: {total_loss/step:.4f}")

    print(f"Epoch {epoch+1} completed. Avg Loss: {total_loss/len(train_loader):.4f}")

NameError: name 'model' is not defined

In [ ]:
from transformers import BartTokenizer, BartForConditionalGeneration

orig_model = BartForConditionalGeneration.from_pretrained("facebook/bart-base").to(device)
orig_tokenizer = BartTokenizer.from_pretrained("facebook/bart-base")
orig_model.eval()


Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

BartForConditionalGeneration(
  (model): BartModel(
    (shared): BartScaledWordEmbedding(50265, 768, padding_idx=1)
    (encoder): BartEncoder(
      (embed_tokens): BartScaledWordEmbedding(50265, 768, padding_idx=1)
      (embed_positions): BartLearnedPositionalEmbedding(1026, 768)
      (layers): ModuleList(
        (0-5): 6 x BartEncoderLayer(
          (self_attn): BartAttention(
            (k_proj): Linear(in_features=768, out_features=768, bias=True)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=768, out_features=3072, bias=True)
          (fc2): Linear(in_features=3072, out_features=768, bias=True)
          (final_layer_n

In [ ]:
from rouge_score import rouge_scorer

def evaluate_model(model, tokenizer, dataset, sample_size=50):
    scorer = rouge_scorer.RougeScorer(["rouge1","rouge2","rougeL"], use_stemmer=True)
    scores = []

    for i in range(sample_size):
        article = dataset[i]["article"]
        reference = dataset[i]["highlights"]

        inputs = tokenizer(article, return_tensors="pt", truncation=True, max_length=512).to(device)
        summary_ids = model.generate(inputs["input_ids"], max_length=128, num_beams=4, early_stopping=True)
        generated = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

        score = scorer.score(reference, generated)
        scores.append(score)

    avg_rouge1 = sum(s["rouge1"].fmeasure for s in scores) / sample_size
    avg_rouge2 = sum(s["rouge2"].fmeasure for s in scores) / sample_size
    avg_rougeL = sum(s["rougeL"].fmeasure for s in scores) / sample_size

    return avg_rouge1, avg_rouge2, avg_rougeL


In [ ]:
# Fine-tuned model (already in memory)
ft_rouge1, ft_rouge2, ft_rougeL = evaluate_model(model, tokenizer, test_dataset)

# Original pretrained BART
orig_rouge1, orig_rouge2, orig_rougeL = evaluate_model(orig_model, orig_tokenizer, test_dataset)

print("Fine-tuned BART:")
print(f"ROUGE-1: {ft_rouge1:.4f}, ROUGE-2: {ft_rouge2:.4f}, ROUGE-L: {ft_rougeL:.4f}")

print("\nOriginal BART (no fine-tuning):")
print(f"ROUGE-1: {orig_rouge1:.4f}, ROUGE-2: {orig_rouge2:.4f}, ROUGE-L: {orig_rougeL:.4f}")


Fine-tuned BART:
ROUGE-1: 0.3132, ROUGE-2: 0.1173, ROUGE-L: 0.2177

Original BART (no fine-tuning):
ROUGE-1: 0.2894, ROUGE-2: 0.1353, ROUGE-L: 0.2056


In [ ]:
save_dir = "./bart_cnn_checkpoint"
model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)

print("Model and tokenizer saved to", save_dir)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model and tokenizer saved to ./bart_cnn_checkpoint


In [ ]:
# Example long paragraph
long_text = """
Artificial intelligence has rapidly evolved over the past decade, transforming industries ranging from healthcare to finance. In hospitals, AI systems assist doctors by analyzing medical images, predicting patient outcomes, and even suggesting personalized treatment plans. In finance, algorithms detect fraudulent transactions and optimize investment strategies. Despite these advances, challenges remain. Ethical concerns about bias, transparency, and accountability continue to spark debate. Governments worldwide are drafting regulations to ensure AI is used responsibly, while researchers are working on explainable AI models that can justify their decisions. As adoption spreads, education systems are also adapting, teaching students not only how to use AI tools but also how to critically evaluate their outputs. The future of AI will likely depend on striking a balance between innovation and responsibility, ensuring that technology serves humanity rather than replacing it.
"""

# --- Fine-tuned BART ---
inputs_ft = tokenizer(long_text, return_tensors="pt", truncation=True, max_length=1024).to(device)
summary_ids_ft = model.generate(
    inputs_ft["input_ids"],
    max_length=400,       # allow longer summaries
    min_length=200,       # force at least this many tokens
    num_beams=4,
    length_penalty=2.0,   # encourage longer outputs
    early_stopping=True
)
summary_ft = tokenizer.decode(summary_ids_ft[0], skip_special_tokens=True)

# --- Original BART ---
inputs_orig = orig_tokenizer(long_text, return_tensors="pt", truncation=True, max_length=1024).to(device)
summary_ids_orig = orig_model.generate(
    inputs_orig["input_ids"],
    max_length=400,
    min_length=200,
    num_beams=4,
    length_penalty=2.0,
    early_stopping=True
)
summary_orig = orig_tokenizer.decode(summary_ids_orig[0], skip_special_tokens=True)

print("Fine-tuned BART summary:\n", summary_ft)
print("\nOriginal BART summary:\n", summary_orig)



Fine-tuned BART summary:
 Artificial intelligence has rapidly evolved over the past decade, transforming industries ranging from healthcare to finance . The future of AI will likely depend on striking a balance between innovation and responsibility . Industry is also adapting, teaching students not only how to use AI tools but also how to critically evaluate their outputs . Technology serves humanity rather than replacing it with artificial machines, experts say . Technology will also be used responsibly, but technology will adapt to adapt to future technology needs in the future, they argue . Software will be the future of the future as technology of humanity, but not humans, they say, they hope to learn from the role that can be built by scientists will also need to be adapted to be used as technology will also adapt to the role of education systems will need to learn to improve on the role will be more than replace it, but will not be replaced by the future for the digital industry 

In [ ]:
!zip -r bart_cnn_checkpoint.zip bart_cnn_checkpoint



  adding: bart_cnn_checkpoint/ (stored 0%)
  adding: bart_cnn_checkpoint/model.safetensors (deflated 8%)
  adding: bart_cnn_checkpoint/tokenizer.json (deflated 82%)
  adding: bart_cnn_checkpoint/config.json (deflated 64%)
  adding: bart_cnn_checkpoint/tokenizer_config.json (deflated 50%)
  adding: bart_cnn_checkpoint/generation_config.json (deflated 47%)


In [ ]:

    model = BartForConditionalGeneration.from_pretrained("./summarization_model", local_files_only=True)

NameError: name 'BartForConditionalGeneration' is not defined

In [ ]:
from google.colab import drive
drive.mount('/content/drive')